# Aligning Xenium breast cancer data to H&E staining image (squidpy + STalign)

In this notebook, we align single cell resolution Xenium spatial transcriptomics data to a corresponding H&E staining image of the same tissue section.

Since this is a point-to-image alignment, we use the STalign/LDDMM backend via `squidpy`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import anndata as ad
import squidpy as sq

plt.rcParams["figure.figsize"] = (12, 10)

## Load H&E image (target)

In [ ]:
image_file = '../xenium_data/Xenium_FFPE_Human_Breast_Cancer_Rep1_he_image.png'
V = plt.imread(image_file)

fig, ax = plt.subplots()
ax.imshow(V)
ax.set_title('H&E image (target)')
print(f"Image shape: {V.shape}")

## Load Xenium cell data (source)

In [ ]:
fname = '../xenium_data/Xenium_FFPE_Human_Breast_Cancer_Rep1_cells.csv.gz'
df = pd.read_csv(fname)

coords_source = np.column_stack([df['x_centroid'].values, df['y_centroid'].values])
adata_source = ad.AnnData(X=np.zeros((len(coords_source), 1)), obs=df)
adata_source.obsm['spatial'] = coords_source

fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2)
ax.set_title('Xenium cell positions (source)')

## Define landmark points

In [ ]:
# Manually defined landmarks (in y,x / row,col order)
pointsI = np.array([[1050., 950.], [700., 2200.], [500., 1550.], [1550., 1840.]])
pointsJ = np.array([[3108., 2100.], [4480., 6440.], [5040., 4200.], [1260., 5320.]])

## Align using STalign (point-to-image)

Note: For this alignment, the H&E image is the source and Xenium positions are being aligned. The squidpy API handles this by aligning the point cloud to the image space.

In [ ]:
# Here we align the H&E (source image) to Xenium cell density (target)
# Then transform the Xenium points using the inverse
sq.experimental.tl.align(
    adata_source,
    V,
    method='lddmm',
    resolution=30.0,
    niter=2000,
    landmark_source=pointsI,
    landmark_target=pointsJ,
    verbose=True,
    sigmaM=0.15,
    sigmaB=0.10,
    sigmaA=0.11,
    epV=10,
)

## Visualize results

In [ ]:
aligned = adata_source.obsm['spatial_aligned']

fig, ax = plt.subplots()
ax.imshow(V)
ax.scatter(aligned[:, 0], aligned[:, 1], s=1, alpha=0.1, label='Xenium aligned')
ax.set_title('Aligned Xenium on H&E')
ax.legend(markerscale=10)

In [ ]:
df_aligned = pd.DataFrame({'aligned_x': aligned[:, 0], 'aligned_y': aligned[:, 1]})
results = pd.concat([df, df_aligned], axis=1)
results.head()